# Customer Purchase Intention Prediction

This notebook is the English, GitHub-ready version of the original Swedish notebook. The project predicts whether an e-commerce customer session will lead to a purchase.

## Notebook-to-Repository Structure

The exploratory notebook has been divided into reusable project files:

- `src/purchase_intention/data.py`: loading, cleaning, encoding, and splitting data.
- `src/purchase_intention/train.py`: model training and final model selection.
- `src/purchase_intention/evaluate.py`: metrics and reports.
- `src/purchase_intention/visualization.py`: plots and confusion matrices.
- `reports/repo_plan.md`: detailed mapping from notebook sections to repository files.

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.append(str(PROJECT_ROOT / "src"))

import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from purchase_intention.data import load_data, clean_data, split_features_target, make_train_validation_test_split
from purchase_intention.train import build_final_model
from purchase_intention.evaluate import evaluate_classifier

## 1. Load The Raw Data

The dataset contains historical customer sessions from an e-commerce website. Each row represents one session, and the target column is `Revenue`.

In [ ]:
raw_df = load_data(PROJECT_ROOT / "data" / "raw" / "project_data.csv")
raw_df.shape, raw_df.head()

## 2. Clean And Encode The Data

The cleaning stage follows the original notebook decisions:

- Remove rows without a valid target value.
- Keep only valid `Weekend` values.
- Convert `Revenue` and `Weekend` to binary values.
- Drop rows with missing `Region`, `Browser`, or `SpecialDay` values.
- Remove invalid month values such as `Turc` and normalize `Sept` to `Sep`.
- Encode month as cyclic sine/cosine features.
- One-hot encode `VisitorType`.
- Remove duplicates and invalid numeric values.

In [ ]:
df = clean_data(raw_df)
df.shape, df.head()

## 3. Explore The Target Balance

The original notebook showed that the target is imbalanced: most sessions do not lead to a purchase. Because of that, accuracy alone is not enough; precision, recall, F1-score, and ROC-AUC are more informative.

In [ ]:
target_distribution = df["Revenue"].value_counts(normalize=True).rename("share")
display(target_distribution)

sns.countplot(x="Revenue", data=df)
plt.title("Distribution of Revenue")
plt.xticks([0, 1], ["No Purchase", "Purchase"])
plt.show()

## 4. Split The Dataset

The project uses the same split as the original notebook: 70% training data, 15% validation data, and 15% test data. Stratification keeps the class balance similar in every split.

In [ ]:
X, y = split_features_target(df)
X_train, X_val, X_test, y_train, y_val, y_test = make_train_validation_test_split(X, y)

X_train.shape, X_val.shape, X_test.shape

## 5. Train The Final Model

The original experiments compared Logistic Regression, balanced Logistic Regression, Random Forest, Gradient Boosting, and several imbalance-handling methods. The selected final model is SMOTE plus tuned Gradient Boosting.

In [ ]:
final_model = build_final_model()
final_model.fit(X_train, y_train)

validation_result = evaluate_classifier(final_model, X_val, y_val)
validation_result.metrics

## 6. Final Test Evaluation

The test set is only used after model selection. In the original notebook, the final model reached about 88.43% accuracy, 60.33% precision, 74.80% recall, 66.79% F1-score, and 92.68% ROC-AUC.

In [ ]:
test_result = evaluate_classifier(final_model, X_test, y_test)
test_result.metrics

## Conclusion

The analysis shows that purchase intention can be predicted reasonably well from historical customer-session behavior. The final model is especially good at finding many of the sessions that actually lead to purchases, which makes it useful as decision support for marketing, personalization, or customer-experience workflows.